# Plastisk deformasjon i en metallkrystall

## Slipsystemer, flytstart og hardning

### Pilotprosjekt for Matematikk 1

Metaller er bygget opp av krystaller. Når en krystall deformeres permanent, skjer mye av deformasjonen ved at atomplan glir i bestemte retninger. En kombinasjon av et **slipplan** og en **slipretning** kalles et slipsystem.

I dette prosjektet undersøker vi tre spørsmål:

1. Hvordan beskriver vi slipplan og slipretninger med lineær algebra?
2. Når begynner ett slipsystem å gli, og hvordan endres spenningen under deformasjon?
3. Hvordan konkurrerer og hardner to slipsystemer hverandre?

Prosjektet bruker en forenklet krystallplastisitetsmodell. Hensikten er å få fram sammenhengen mellom krystallgeometri, spenning, plastisk deformasjon og hardning. Modellen er ikke en full modell for et polykrystallinsk metall.

### Læringsmål

Etter prosjektet skal du kunne

- bruke en basis og en dualbasis til å beskrive retninger og plan,
- kontrollere om en retning ligger i et plan,
- gjennomføre basisbytte mellom krystall- og laboratoriekoordinater,
- beregne oppløst skjærspenning med matriser og vektorer,
- finne flytstart i en enkel elastisk-plastisk modell,
- løse en lineær førsteordens ODE analytisk og med Eulers metode,
- implementere et koblet, stykkevis definert ODE-system,
- tolke egenhardning og latent hardning.

### Modellnivå

Den opprinnelige inspirasjonen var en komposittmodell med celleinteriører og cellevegger, der kryssglidning og dislokasjonsklatring gir ulike gjenopprettingsmekanismer. Den modellen er for omfattende for Matematikk 1. Her bruker vi i stedet en minimal modell som beholder de viktigste ideene: orienteringsavhengig slip, en kinetisk flytlov og utvikling av materialets motstand mot videre slip.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Del A: Krystallgeometri og lineær algebra

## A.1 Gitterbasis og dualbasis

La krystallgitterets basisvektorer være kolonnene i matrisen

$$
B=\begin{pmatrix}|&|&|\\ b_1&b_2&b_3\\ |&|&|\end{pmatrix}.
$$

Dualbasisen $b_1^*,b_2^*,b_3^*$ er definert ved

$$b_i^*\cdot b_j=\delta_{ij}.$$

Dersom dualbasisvektorene samles som kolonner i $B^*$, gjelder

$$
B^*=B^{-T}.
$$

Et plan med Miller-indekser $(hkl)$ har en normalretning

$$
n_{rå}=h b_1^*+k b_2^*+l b_3^*.
$$

En krystallretning $[uvw]$ representeres av

$$
s_{rå}=u b_1+v b_2+w b_3.
$$

Vi normaliserer deretter vektorene:

$$n=\frac{n_{rå}}{\|n_{rå}\|},\qquad
s=\frac{s_{rå}}{\|s_{rå}\|}.
$$

## Oppgave A1: Beregn dualbasisen

Vi bruker først en enkel kubisk basis med gitterkonstant $a=1$:

$$B=I.$$

Beregn $B^*$ og kontroller numerisk at

$$B^{*T}B=I.$$

Deretter kan du prøve den primitive FCC-basis som er gitt i kodecellen.

In [ ]:
a = 1.0

B_kubisk = a*np.eye(3)

B_fcc = (a/2)*np.array([
    [0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0]
])

B = B_kubisk

B_dual = ...
print("B* =
", B_dual)
print("Kontroll B*^T B =
", ...)

## A.2 Slipsystemer i FCC

For mange FCC-metaller, blant annet aluminium, kobber og nikkel, brukes slipfamilien

$$\{111\}\langle110\rangle.$$

Dette betyr at slipplanet tilhører familien $\{111\}$, mens slipretningen tilhører familien $\langle110\rangle$.

Et gyldig slipsystem må oppfylle

$$n\cdot s=0,$$

fordi slipretningen må ligge i slipplanet.

## Oppgave A2: Kontroller slipsystemer

Bruk planet $(111)$ og retningene

$$[1\bar10],\qquad [10\bar1],\qquad [01\bar1].$$

Beregn normalvektoren og retningene. Kontroller at alle tre retningene ligger i planet.

Undersøk også retningen $[110]$. Ligger den i planet $(111)$?

In [ ]:
def normal_fra_miller(hkl, B_dual):
    n_raa = B_dual @ np.array(hkl, dtype=float)
    return n_raa / np.linalg.norm(n_raa)


def retning_fra_indekser(uvw, B):
    s_raa = B @ np.array(uvw, dtype=float)
    return s_raa / np.linalg.norm(s_raa)


n = normal_fra_miller([1, 1, 1], B_dual)

retninger = {
    "[1 -1 0]": [1, -1, 0],
    "[1 0 -1]": [1, 0, -1],
    "[0 1 -1]": [0, 1, -1],
    "[1 1 0]": [1, 1, 0]
}

for navn, uvw in retninger.items():
    s = retning_fra_indekser(uvw, B)
    print(navn, "n dot s =", ...)

## A.3 Basisbytte

Krystallens akser trenger ikke være parallelle med prøvelegemets akser. La $Q$ være en rotasjonsmatrise som omformer krystallkoordinater til laboratoriekoordinater:

$$v_{lab}=Qv_{krystall}.$$

Dermed blir

$$n_{lab}=Qn_{krystall},\qquad s_{lab}=Qs_{krystall}.$$

Spenning er representert ved en symmetrisk matrise $\sigma$. Den oppløste skjærspenningen på slipsystemet er

$$
\boxed{\tau=s_{lab}^T\sigma_{lab}n_{lab}.}
$$

Den samme beregningen kan utføres i krystallkoordinater ved å transformere spenningen:

$$
sigma_{krystall}=Q^T\sigma_{lab}Q.
$$

## Oppgave A3: Rotasjonsmatrise

Vi roterer krystallen først en vinkel $\theta$ om laboratoriets $z$-akse og deretter en vinkel $\varphi$ om $y$-aksen.

Fullfør rotasjonsmatrisene og kontroller at

$$Q^TQ=I,\qquad \det Q=1.$$

In [ ]:
theta = np.deg2rad(25.0)
phi = np.deg2rad(35.0)

Qz = np.array([
    [np.cos(theta), -np.sin(theta), 0.0],
    [np.sin(theta),  np.cos(theta), 0.0],
    [0.0,            0.0,           1.0]
])

Qy = np.array([
    [ np.cos(phi), 0.0, np.sin(phi)],
    [0.0,          1.0, 0.0],
    [-np.sin(phi), 0.0, np.cos(phi)]
])

Q = ...

print("Q^T Q =
", ...)
print("det(Q) =", ...)

## Oppgave A4: Oppløst skjærspenning

Prøven utsettes for en enakset strekkspenning i laboratoriets $x$-retning:

$$
\sigma_{lab}=
\begin{pmatrix}
\sigma_0&0&0\\
0&0&0\\
0&0&0
\end{pmatrix}.
$$

Bruk $\sigma_0=100$ MPa. Beregn $\tau$ for flere FCC-slipsystemer.

Kontroller resultatet på to måter:

1. roter $n$ og $s$ til laboratoriekoordinater,
2. roter $\sigma$ til krystallkoordinater.

De to beregningene skal gi samme svar.

In [ ]:
sigma0 = 100.0  # MPa
sigma_lab = np.diag([sigma0, 0.0, 0.0])

slipsystemer = [
    ([1, 1, 1], [1, -1, 0]),
    ([1, 1, 1], [1, 0, -1]),
    ([1, 1, -1], [1, -1, 0]),
    ([1, -1, 1], [1, 0, -1])
]

resultater = []

for hkl, uvw in slipsystemer:
    n_k = normal_fra_miller(hkl, B_dual)
    s_k = retning_fra_indekser(uvw, B)

    # Kontroller først at retningen ligger i planet.
    gyldig = ...

    n_lab = ...
    s_lab = ...
    tau_lab = ...

    sigma_k = ...
    tau_k = ...

    resultater.append((hkl, uvw, gyldig, tau_lab, tau_k))

for rad in resultater:
    print(rad)

## Oppgave A5: Hvilket system aktiveres først?

Et slipsystem begynner å gli når den absolutte oppløste skjærspenningen når en kritisk verdi $g_0$:

$$|\tau_i|\ge g_0.$$

Bruk $g_0=30$ MPa.

1. Hvilket av slipsystemene har størst $|\tau_i|$?
2. Ved hvilken påført strekkspenning $\sigma_0$ forventer du at dette systemet aktiveres?
3. Gjenta for noen andre krystallorienteringer $Q$.
4. Forklar hvorfor to krystaller av samme metall kan begynne å gli ved forskjellig påført spenning.

# Del B: Ett slipsystem med hardning

Vi studerer nå bare ett aktivt slipsystem under positiv, monoton skjærbelastning.

La

- $\Gamma(t)$ være påført total skjærdeformasjon,
- $\gamma(t)$ være plastisk skjærdeformasjon fra slip,
- $\Gamma-\gamma$ være elastisk skjærdeformasjon,
- $\mu$ være skjærmodulen,
- $\tau(t)$ være skjærspenningen.

Da bruker vi den elastiske sammenhengen

$$
\tau=\mu(\Gamma-\gamma).
$$

Materialets motstand mot slip øker med plastisk deformasjon:

$$
g=g_0+H\gamma.
$$

Her er $g_0$ den opprinnelige kritiske skjærspenningen og $H$ en hardningsmodul.

Vi bruker flytloven

$$
\dot\gamma=
\frac{1}{\eta}\langle\tau-g\rangle,
\qquad
\langle x\rangle=\max(x,0).
$$

Parameteren $\eta$ beskriver hvor raskt slip utvikles når spenningen overstiger motstanden.

## Oppgave B1: Flytstart

Vi påfører deformasjonen med konstant hastighet:

$$\Gamma(t)=\dot\Gamma t.$$

Før flytstart er $\gamma=0$, slik at

$$\tau=\mu\dot\Gamma t.$$

Vis at flyt starter ved

$$
\boxed{t_y=\frac{g_0}{\mu\dot\Gamma}.}
$$

Hva er totaldeformasjonen $\Gamma_y$ ved flytstart?

## Oppgave B2: ODE etter flytstart

Når $t>t_y$, sett inn

$$\tau=\mu(\dot\Gamma t-\gamma),\qquad g=g_0+H\gamma$$

i flytloven. Vis at

$$
\boxed{
\dot\gamma+\frac{\mu+H}{\eta}\gamma
=
\frac{\mu\dot\Gamma}{\eta}t-\frac{g_0}{\eta}.
}
$$

Dette er en lineær førsteordens ODE med en tidsavhengig høyreside. Løs den for hånd med begynnelsesbetingelsen

$$\gamma(t_y)=0.$$

## Oppgave B3: Numerisk modell

Bruk skalerte parameterverdier

$$
\mu=80,\qquad g_0=20,\qquad H=12,
\qquad \eta=4,\qquad \dot\Gamma=0.01.
$$

Fullfør modellen og Eulers metode. Beregn $\gamma(t)$, $\tau(t)$ og $g(t)$.

In [ ]:
mu = 80.0
g0 = 20.0
H = 12.0
eta = 4.0
Gamma_dot = 0.01


def macaulay(x):
    return np.maximum(x, 0.0)


def ett_slipsystem(t, gamma):
    Gamma = Gamma_dot*t
    tau = ...
    g = ...
    dgamma = ...
    return dgamma


def euler_skalar(f, y0, T, h):
    N = int(round(T/h))
    t = np.linspace(0.0, N*h, N + 1)
    y = np.zeros(N + 1)
    y[0] = y0

    for n in range(N):
        y[n + 1] = ...

    return t, y


t_B, gamma_B = euler_skalar(ett_slipsystem, 0.0, T=80.0, h=0.02)
Gamma_B = Gamma_dot*t_B
tau_B = ...
g_B = ...

In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True)

ax[0].plot(Gamma_B, tau_B, label="Spenning")
ax[0].plot(Gamma_B, g_B, "--", label="Motstand mot slip")
ax[0].set_ylabel("Skalert skjærspenning")
ax[0].legend()
ax[0].grid()

ax[1].plot(Gamma_B, gamma_B)
ax[1].set_xlabel("Total skjærdeformasjon")
ax[1].set_ylabel("Plastisk slip")
ax[1].grid()

plt.show()

## Oppgave B4: Tolk spenning–deformasjonskurven

1. Marker den beregnede flytstarten.
2. Hvilken del av kurven er hovedsakelig elastisk?
3. Hvorfor fortsetter spenningen å øke etter flytstart?
4. Hva skjer dersom $H=0$?
5. Hva skjer når $\eta$ økes eller reduseres?
6. Sammenlign Euler-løsningen med håndløsningen fra B2.

# Del C: To koblede slipsystemer

Vi modellerer to mulige slipsystemer. Hvert system har

- plastisk slip $\gamma_i$,
- kritisk motstand $g_i$,
- en orienteringsfaktor $m_i$.

For en gitt lastretning beskriver $m_i$ hvor stor del av den makroskopiske skjærspenningen som virker på system $i$.

Vi bruker den forenklede koblingen

$$
\tau=\mu\left(
\Gamma-m_1\gamma_1-m_2\gamma_2
\right),
$$

og oppløste skjærspenninger

$$
\tau_1=m_1\tau,
\qquad
\tau_2=m_2\tau.
$$

Denne modellen er bevisst enkel. Den erstatter den fullstendige tensorielle sammenhengen mellom plastisk tøyning og flere Schmid-tensorer med to orienteringsfaktorer.

Flytlovene er

$$
\dot\gamma_i
=
\frac{1}{\eta_i}
\langle |\tau_i|-g_i\rangle
\operatorname{sign}(\tau_i).
$$

Hardningen kobles gjennom

$$
\begin{pmatrix}
\dot g_1\\
\dot g_2
\end{pmatrix}
=
H
\begin{pmatrix}
1&q\\
q&1
\end{pmatrix}
\begin{pmatrix}
|\dot\gamma_1|\\
|\dot\gamma_2|
\end{pmatrix}.
$$

Parameteren $q$ beskriver latent hardning, altså hvordan slip på ett system øker motstanden på det andre.

## Oppgave C1: Hardningsmatrisen

Undersøk hardningsmatrisen

$$K=H\begin{pmatrix}1&q\\q&1\end{pmatrix}.$$

1. Hva betyr diagonal- og ikke-diagonalelementene?
2. Hva skjer når $q=0$?
3. Hva skjer når $q=1$?
4. Finn egenverdiene og egenvektorene til $K$.
5. For hvilke verdier av $q$ er begge egenverdiene ikke-negative?

Dette er en matematisk måte å undersøke om modellen kan gi negative hardningsretninger.

In [ ]:
H = 12.0
q = 1.2

K_hardning = H*np.array([
    [1.0, q],
    [q, 1.0]
])

print("Hardningsmatrise:
", K_hardning)
print("Egenverdier:", ...)
print("Egenvektorer:
", ...)

## Oppgave C2: Implementer vektor-ODE-en

Bruk tilstandsvektoren

$$x=(\gamma_1,\gamma_2,g_1,g_2)^T.$$

Vi velger først

$$m_1=0.48,\qquad m_2=0.36,$$

slik at system 1 er gunstigere orientert enn system 2.

In [ ]:
mu = 80.0
eta1 = 3.0
eta2 = 3.0
Gamma_dot = 0.01
m1 = 0.48
m2 = 0.36
H = 12.0
q = 0.8


def signert_overstress(tau, g):
    return np.sign(tau)*macaulay(abs(tau) - g)


def to_slipsystemer(t, x):
    gamma1, gamma2, g1, g2 = x

    Gamma = Gamma_dot*t
    tau = ...
    tau1 = ...
    tau2 = ...

    dgamma1 = ...
    dgamma2 = ...

    dg1 = ...
    dg2 = ...

    return np.array([dgamma1, dgamma2, dg1, dg2])


def euler_system(f, x0, T, h):
    N = int(round(T/h))
    t = np.linspace(0.0, N*h, N + 1)
    X = np.zeros((N + 1, len(x0)))
    X[0] = x0

    for n in range(N):
        X[n + 1] = ...

    return t, X


x0 = np.array([0.0, 0.0, 9.0, 9.0])
t_C, X_C = euler_system(to_slipsystemer, x0, T=120.0, h=0.01)

gamma1 = X_C[:, 0]
gamma2 = X_C[:, 1]
g1 = X_C[:, 2]
g2 = X_C[:, 3]
Gamma_C = Gamma_dot*t_C

tau_C = ...
tau1_C = ...
tau2_C = ...

In [ ]:
fig, ax = plt.subplots(3, 1, sharex=True, figsize=(7, 9))

ax[0].plot(Gamma_C, tau_C, label="Makrospenning")
ax[0].set_ylabel("Spenning")
ax[0].grid()
ax[0].legend()

ax[1].plot(Gamma_C, gamma1, label="gamma1")
ax[1].plot(Gamma_C, gamma2, label="gamma2")
ax[1].set_ylabel("Plastisk slip")
ax[1].grid()
ax[1].legend()

ax[2].plot(Gamma_C, g1, label="g1")
ax[2].plot(Gamma_C, g2, label="g2")
ax[2].set_xlabel("Total deformasjon")
ax[2].set_ylabel("Motstand mot slip")
ax[2].grid()
ax[2].legend()

plt.show()

## Oppgave C3: Hvilket system aktiveres først?

1. Finn første tidspunkt der $\dot\gamma_1>0$.
2. Finn første tidspunkt der $\dot\gamma_2>0$.
3. Forklar resultatet ved hjelp av $m_1,m_2,g_1,g_2$.
4. Kan system 2 forbli inaktivt gjennom hele forsøket?
5. Hvordan påvirker latent hardning aktiveringen av system 2?

In [ ]:
# Beregn slipphastighetene langs den ferdige løsningen.
dgamma1_serie = []
dgamma2_serie = []

for t_i, x_i in zip(t_C, X_C):
    dx_i = to_slipsystemer(t_i, x_i)
    dgamma1_serie.append(dx_i[0])
    dgamma2_serie.append(dx_i[1])

dgamma1_serie = np.array(dgamma1_serie)
dgamma2_serie = np.array(dgamma2_serie)

# Finn første indeks der hver hastighet er større enn en liten toleranse.

## Oppgave C4: Egenhardning og latent hardning

Gjenta simuleringen for

$$q=0,\qquad q=0.5,\qquad q=1.0.$$

Sammenlign

- flytstart for system 2,
- total slip på hvert system,
- sluttverdiene til $g_1$ og $g_2$,
- spenning–deformasjonskurven.

Forklar hvorfor $q>1$ bør brukes med forsiktighet i akkurat den symmetriske hardningsmatrisen. Se resultatet fra C1.

## Oppgave C5: Orientering

Endre orienteringsfaktorene $m_1$ og $m_2$.

Undersøk minst disse tilfellene:

1. $m_1>m_2$,
2. $m_1=m_2$,
3. $m_1<m_2$.

Koble resultatene tilbake til del A. I en mer fullstendig modell ville $m_i$ bli beregnet direkte fra slipsystemenes normaler, retninger, krystallorientering og spenningsmatrise.

# Del D: Modellkritikk og videreføring

## D1. Hva er forenklet?

Diskuter minst fire punkter:

- Krystallen har bare ett eller to slipsystemer i modellen.
- Orienteringsfaktorene erstatter en full tensorberegning.
- Lasten er monoton og fortegnet skifter ikke.
- Flytloven er lineær i overstresset.
- Temperatur inngår ikke eksplisitt.
- Krystallorienteringen oppdateres ikke når krystallen deformeres.
- Dislokasjonstetthet modelleres ikke direkte.
- Korn og korngrenser er ikke med.
- Tvillingdannelse er ikke med.

## D2. Forbindelse til den opprinnelige komposittmodellen

Den opprinnelige referansemodellen skiller mellom et mykere celleinteriør og hardere cellevegger. Hver del har lokal spenning og lokal styrke. Plastisk flyt i de to delene kobles gjennom

- kompatibilitet av total deformasjon,
- mekanisk likevekt,
- avsetning og fjerning av dislokasjoner,
- kryssglidning i celleinteriøret,
- diffusjonsstyrt klatring i celleveggene.

Forklar hvilke av disse ideene som har en analog i den forenklede modellen, og hvilke som mangler helt.

## Videreføring til Matematikk 2 og 3

I senere emner kan modellen utvides til funksjoner fra $\mathbb R^n$ til $\mathbb R^m$, tensorregning og romlig varierende deformasjon. En full krystallplastisitetsmodell bruker Schmid-tensorer

$$
P_i=\frac12(s_i\otimes n_i+n_i\otimes s_i)
$$

og bygger plastisk tøyning fra bidragene til alle aktive slipsystemer. I en romlig modell blir spenning, tøyning og slip avhengige av posisjon.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan dualbasisen brukes til å beskrive slipplan,
2. hvordan du kontrollerte at en slipretning ligger i et plan,
3. hvordan basisbytte påvirket den oppløste skjærspenningen,
4. hvordan flytstart ble funnet i ettsystemmodellen,
5. hvordan elastisitet, slip og hardning formet spenning–deformasjonskurven,
6. hvordan de to slipsystemene konkurrerte,
7. hvordan latent hardning påvirket systemet,
8. hvilke begrensninger ved modellen som er viktigst.

## Referanser for prosjektutviklingen

- W. D. Nix, J. C. Gibeling og D. A. Hughes, *Time-Dependent Deformation of Metals*.
- Callister og Rethwisch, oversikt over slipsystemer i FCC, BCC og HCP.

Studentene trenger ikke lese forskningsartikkelen for å gjennomføre prosjektet.